In [ ]:
from datetime import datetime, timezone
import time
import hmac
import hashlib
import requests
import pytz
import pandas as pd
from urllib.parse import urlencode
import os

BASE = "https://api.bitso.com"

from google.colab import userdata

API_KEY = userdata.get("BITSO_API_KEY")
API_SECRET = userdata.get("BITSO_API_SECRET")

if not API_KEY or not API_SECRET:
    raise ValueError("No se encontraron credenciales en los Secretos de Colab.")

# -----------------------------
# Nonce monotónico
# -----------------------------
_last_nonce = 0
def nonce_ms() -> str:
    global _last_nonce
    n = int(time.time() * 1000)
    if n <= _last_nonce:
        n = _last_nonce + 1
    _last_nonce = n
    return str(n)

# -----------------------------
# Signed request
# -----------------------------
def bitso_signed_request(method: str, request_path: str, params=None, body: str = ""):
    method = method.upper()
    params = params or {}

    if params:
        qs = urlencode(sorted(params.items()))
        path_with_qs = f"{request_path}?{qs}"
    else:
        path_with_qs = request_path

    nonce = nonce_ms()
    message = f"{nonce}{method}{path_with_qs}{body}"
    signature = hmac.new(
        API_SECRET.encode("utf-8"),
        message.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

    headers = {
        "Authorization": f"Bitso {API_KEY}:{nonce}:{signature}",
        "Content-Type": "application/json",
    }

    url = BASE + path_with_qs
    r = requests.request(method, url, headers=headers, timeout=30)

    if r.status_code >= 400:
        try:
            err = r.json()
        except Exception:
            err = r.text
        raise RuntimeError(f"HTTP {r.status_code} error for {url}\nResponse: {err}")

    return r.json()

def bitso_signed_get(request_path: str, params=None):
    return bitso_signed_request("GET", request_path, params=params, body="")

# -----------------------------
# Public GET
# -----------------------------
def bitso_public_get(request_path, params=None):
    params = params or {}
    r = requests.get(BASE + request_path, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def get_price_last(book: str) -> float:
    j = bitso_public_get("/api/v3/ticker/", {"book": book})
    if not j.get("success", False):
        raise RuntimeError(f"Ticker failed for {book}: {j}")
    return float(j["payload"]["last"])

# -----------------------------
# Helpers: balances + pricing
# -----------------------------
ASSETS = ["ATOM", "ETH", "DOT", "SOL", "ADA"]

def get_balance_total(balance_json, currency: str) -> float:
    balances = balance_json["payload"]["balances"]
    c = currency.lower()
    row = next((b for b in balances if b["currency"].lower() == c), None)
    return float(row["total"]) if row else 0.0

def get_price_mxn_usd(asset: str):
    """
    Devuelve (price_mxn, price_usd, fx_usd_mxn, pricing_method)
    pricing_method: 'direct_mxn', 'usd_fx', o 'missing'
    """
    asset_l = asset.lower()

    # FX lo intentamos siempre (para que no quede NaN)
    fx = None
    try:
        fx = get_price_last("usd_mxn")
    except Exception:
        fx = None

    price_mxn = None
    price_usd = None
    method = "missing"

    # 1) Intentar book directo asset_mxn
    try:
        price_mxn = get_price_last(f"{asset_l}_mxn")
        method = "direct_mxn"
    except Exception:
        # 2) Si no hay, usar asset_usd + usd_mxn
        try:
            price_usd = get_price_last(f"{asset_l}_usd")
            if fx is None:
                raise RuntimeError("No pude obtener usd_mxn para convertir a MXN.")
            price_mxn = price_usd * fx
            method = "usd_fx"
        except Exception:
            return (None, None, fx, "missing")

    # Si conseguimos MXN directo, intentar USD (opcional)
    if price_usd is None:
        try:
            price_usd = get_price_last(f"{asset_l}_usd")
        except Exception:
            price_usd = None

    return (price_mxn, price_usd, fx, method)

def build_portfolio_snapshot(balance_json, assets=ASSETS):
    tz = pytz.timezone("America/Monterrey")
    fecha = datetime.now(tz).date().isoformat()
    pulled_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

    rows = []
    for asset in assets:
        total = get_balance_total(balance_json, asset)
        price_mxn, price_usd, fx, method = get_price_mxn_usd(asset)

        value_mxn = total * price_mxn if (total is not None and price_mxn is not None) else None
        value_usd = total * price_usd if (total is not None and price_usd is not None) else None

        rows.append({
            "fecha": fecha,
            "cuenta": "bitso",
            "asset": asset,
            "total": total,
            "price_usd": price_usd,
            "price_mxn": price_mxn,
            "fx_usd_mxn": fx,
            "value_usd": value_usd,
            "value_mxn": value_mxn,
            "pricing_method": method,
            "pulled_at_utc": pulled_at_utc
        })

    df = pd.DataFrame(rows)

    # Totales (por si quieres ver el “gran total”)
    total_mxn = df["value_mxn"].dropna().sum()
    total_usd = df["value_usd"].dropna().sum()

    return df, total_mxn, total_usd

# -----------------------------
# EJECUCIÓN
# -----------------------------
balance_json = bitso_signed_get("/api/v3/balance/")

df_assets, total_mxn, total_usd = build_portfolio_snapshot(balance_json, ASSETS)

print(f"Total portafolio (MXN): {total_mxn:,.2f}")
print(f"Total portafolio (USD): {total_usd:,.2f}")

df_assets